In [ ]:
# test_incremental_v2 — v2标签 (C1归入不重定位) + 5-fold CV
import pickle, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
LAB  = pd.read_csv(BASE + "cell2024_model_single_subst.csv")
LAB["KEY"] = LAB["Gene"].astype(str) + "||" + LAB["Variant"].astype(str)

# ===== v2 标签 =====
LAB["reloc_v2"] = LAB["Mislocalized"].astype(int)
LAB.loc[LAB["label_5class"] == "C1_no_reloc", "reloc_v2"] = 0
print(f"v2标签: 0={(LAB['reloc_v2']==0).sum()}, 1={(LAB['reloc_v2']==1).sum()}")

esm = pickle.load(open(BASE + "phase4_esm2_local_delta.pkl", "rb"))
pup = pickle.load(open("/mnt/volume6/czj/labLGN/LabLZ/pups_trial/pups_full_delta.pkl", "rb"))

rows = []
for _, r in LAB.iterrows():
    k = r["KEY"]
    if pd.isna(r["reloc_v2"]):
        continue
    e, p = esm.get(k), pup.get(k)
    if e is None or p is None:
        continue
    rows.append((r["Gene"], int(r["reloc_v2"]), e, p))

genes = np.array([r[0] for r in rows])
y     = np.array([r[1] for r in rows])
E     = np.vstack([r[2] for r in rows])   # ESM2 local delta  (n, 1280)
P     = np.vstack([r[3] for r in rows])   # PUPS delta        (n, 29)

print(f"样本 n={len(y)}, 正={int(y.sum())}, 基因={len(set(genes))}")

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
def evaluate(X, name):
    pipe = make_pipeline(SimpleImputer(), StandardScaler(),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))
    oof = cross_val_predict(pipe, X, y, cv=cv, groups=genes, method="predict_proba")[:, 1]
    auroc = roc_auc_score(y, oof)
    auprc = average_precision_score(y, oof)
    print(f"  {name:26s} AUROC={auroc:.3f}  AUPRC={auprc:.3f}")
    return auroc

print("=== v2 增量测试 ===")
evaluate(P, "PUPS only")
b = evaluate(E, "ESM2-delta only")
c = evaluate(np.hstack([E, P]), "ESM2-delta + PUPS")
print(f"\n增量 Δ = {c - b:+.3f} → {'PUPS 有加分' if c - b > 0.02 else 'PUPS 无明显增量'}")

v2标签: 0=1956, 1=223
样本 n=2179, 正=223, 基因=871
=== v2 增量测试 ===
  PUPS only                  AUROC=0.515  AUPRC=0.128
  ESM2-delta only            AUROC=0.533  AUPRC=0.121
  ESM2-delta + PUPS          AUROC=0.543  AUPRC=0.126

增量 Δ = +0.010 → PUPS 无明显增量
